In [2]:
"""
================================================================================
FEATURES FOR CONGRESSIONAL TRADE ANOMALY DETECTION
================================================================================


This script creates features for detecting anomalous trades by US Congress members.

Date: 2025
================================================================================
"""

import pandas as pd
import numpy as np
from datetime import timedelta
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

INPUT_PATH = r'C:\Users\sebib\Documents\GitHub\US_Congress\data\congress_trades\congress_trades_with_committees.parquet'
OUTPUT_PATH = r'C:\Users\sebib\Documents\GitHub\US_Congress\data\congress_trades\congress_trades_anomaly_features.parquet'

# Windows for rolling calculations
COORDINATION_WINDOW_DAYS = 7  # Window to detect coordinated trades
HISTORICAL_WINDOW_TRADES = 50  # Min trades for historical features

# ============================================================================
# LOAD DATA
# ============================================================================

print("="*70)
print("FEATURE ENGINEERING FOR ANOMALY DETECTION")
print("="*70)

df = pd.read_parquet(INPUT_PATH)
print(f"\nLoaded: {len(df):,} trades")

# Ensure datetime
df['trade_date'] = pd.to_datetime(df['trade_date'])
df['Filed'] = pd.to_datetime(df['Filed'])
df = df.sort_values(['trade_date', 'Name']).reset_index(drop=True)

# ============================================================================
# 1. POLITICIAN CHARACTERISTICS (Static per politician)
# ============================================================================
print("\n[1/8] Politician characteristics...")

# 1.1 Is Senator (vs Representative)
df['is_senator'] = (df['Chamber'] == 'Senate').astype(int)

# 1.2 Committee Role Power
role_power = {
    'Chair': 3,
    'Ranking Member': 2,
    'Vice Chair': 2,
    'Member': 1
}
df['committee_role_power'] = df['committee_role'].map(role_power).fillna(1)

# 1.3 Is Chair or Ranking Member (high power positions)
df['is_chair_or_ranking'] = df['committee_role'].isin(['Chair', 'Ranking Member']).astype(int)

# 1.4 Committee Topic relevance mapping (for later use with sectors)
high_info_committees = [
    'finance', 'banking', 'commerce', 'energy', 'health', 
    'appropriations', 'budget', 'intelligence', 'armed services'
]
df['is_high_info_committee'] = df['committee_topic'].str.lower().isin(high_info_committees).astype(int)

# 1.5 Number of committees per politician (proxy for information access)
committees_per_politician = df.groupby('BioGuideID')['committee_id'].nunique().reset_index()
committees_per_politician.columns = ['BioGuideID', 'n_committees']
df = df.merge(committees_per_politician, on='BioGuideID', how='left')

print(f"   ✓ is_senator, is_chair_or_ranking, is_high_info_committee, n_committees")

# ============================================================================
# 2. TRADE CHARACTERISTICS (Per trade)
# ============================================================================
print("\n[2/8] Trade characteristics...")

# 2.1 Disclosure Delay (days between trade and filing)
df['disclosure_delay'] = (df['Filed'] - df['trade_date']).dt.days
df['disclosure_delay'] = df['disclosure_delay'].clip(lower=0, upper=365)

# 2.2 Is Late Disclosure (> 45 days, legal limit is 45 days)
df['is_late_disclosure'] = (df['disclosure_delay'] > 45).astype(int)

# 2.3 Parse Trade Size
def parse_trade_size(size_str):
    """Extract midpoint of trade size range"""
    if pd.isna(size_str):
        return np.nan
    size_str = str(size_str).replace('$', '').replace(',', '').strip()
    
    if ' - ' in size_str:
        parts = size_str.split(' - ')
        try:
            low = float(parts[0])
            high = float(parts[1])
            return (low + high) / 2
        except:
            return np.nan
    elif 'Over' in size_str or '>' in size_str:
        try:
            val = float(size_str.replace('Over', '').replace('>', '').strip())
            return val * 1.5
        except:
            return np.nan
    else:
        try:
            return float(size_str)
        except:
            return np.nan

df['trade_size_mid'] = df['Trade_Size_USD'].apply(parse_trade_size)

# 2.4 Trade Size Categories
df['is_large_trade'] = (df['trade_size_mid'] >= 50000).astype(int)
df['is_very_large_trade'] = (df['trade_size_mid'] >= 250000).astype(int)

# 2.5 Is Purchase (vs Sale)
df['is_purchase'] = df['Transaction'].str.contains('Purchase', case=False, na=False).astype(int)

# 2.6 Is Full Sale (complete liquidation)
df['is_full_sale'] = df['Transaction'].str.contains('Full', case=False, na=False).astype(int)

# 2.7 Is Contrarian (trading against recent momentum)
df['is_contrarian'] = (
    ((df['is_purchase'] == 1) & (df['momentum_20d'] < -0.05)) |
    ((df['is_purchase'] == 0) & (df['momentum_20d'] > 0.05))
).astype(int)

# 2.8 Strong Contrarian (more extreme)
df['is_strong_contrarian'] = (
    ((df['is_purchase'] == 1) & (df['momentum_20d'] < -0.10)) |
    ((df['is_purchase'] == 0) & (df['momentum_20d'] > 0.10))
).astype(int)

print(f"   ✓ disclosure_delay, is_late_disclosure, trade_size_mid")
print(f"   ✓ is_large_trade, is_very_large_trade, is_purchase, is_full_sale")
print(f"   ✓ is_contrarian, is_strong_contrarian")

# ============================================================================
# 3. POLITICIAN HISTORICAL BEHAVIOR (Rolling features per politician)
# ============================================================================
print("\n[3/8] Politician historical behavior...")

df = df.sort_values(['BioGuideID', 'trade_date']).reset_index(drop=True)

politician_features = []

for bioguide_id, group in df.groupby('BioGuideID'):
    group = group.sort_values('trade_date').copy()
    n_trades = len(group)
    
    # 3.1 Cumulative trade count
    group['politician_trade_count'] = range(1, n_trades + 1)
    
    # 3.2 Days since last trade
    group['days_since_last_trade'] = group['trade_date'].diff().dt.days.fillna(999)
    
    # 3.3 Rolling win rate
    if 'car_ff3_30d' in group.columns:
        group['politician_rolling_win_rate'] = (
            (group['car_ff3_30d'] > 0)
            .shift(1)
            .expanding()
            .mean()
        )
    else:
        group['politician_rolling_win_rate'] = np.nan
    
    # 3.4 Rolling average CAR
    if 'car_ff3_30d' in group.columns:
        group['politician_rolling_avg_car'] = (
            group['car_ff3_30d']
            .shift(1)
            .expanding()
            .mean()
        )
    else:
        group['politician_rolling_avg_car'] = np.nan
    
    # 3.5 Rolling trade frequency (last 6 months)
    group['politician_6m_trade_count'] = (
        group.set_index('trade_date')['trade_id']
        .rolling('180D', min_periods=1)
        .count()
        .values
    )
    
    # 3.6 Average monthly trades
    group['politician_avg_monthly_trades'] = group['politician_trade_count'] / (
        (group['trade_date'] - group['trade_date'].min()).dt.days / 30 + 1
    )
    
    # 3.7 Trade size relative to history
    group['politician_avg_trade_size'] = (
        group['trade_size_mid']
        .shift(1)
        .expanding()
        .mean()
    )
    group['trade_size_vs_history'] = group['trade_size_mid'] / group['politician_avg_trade_size'].replace(0, np.nan)
    
    # 3.8 Is new ticker for this politician
    group['ticker_trade_count'] = group.groupby('Ticker').cumcount()
    group['is_new_ticker'] = (group['ticker_trade_count'] == 0).astype(int)
    
    politician_features.append(group)

df = pd.concat(politician_features, ignore_index=True)

print(f"   ✓ politician_trade_count, days_since_last_trade")
print(f"   ✓ politician_rolling_win_rate, politician_rolling_avg_car")
print(f"   ✓ politician_6m_trade_count, trade_size_vs_history, is_new_ticker")

# ============================================================================
# 4. COORDINATION FEATURES (Collective anomalies)
# ============================================================================
print("\n[4/8] Coordination features...")
print("      This may take a few minutes...")

df = df.sort_values('trade_date').reset_index(drop=True)

coordination_features = []

for idx, row in df.iterrows():
    trade_date = row['trade_date']
    ticker = row['Ticker']
    politician = row['BioGuideID']
    is_buy = row['is_purchase']
    committee = row['committee_id']
    party = row['Party']
    
    window_start = trade_date - timedelta(days=COORDINATION_WINDOW_DAYS)
    window_end = trade_date + timedelta(days=COORDINATION_WINDOW_DAYS)
    
    # Trades in same ticker within window (excluding self)
    window_mask = (
        (df['trade_date'] >= window_start) & 
        (df['trade_date'] <= window_end) &
        (df['Ticker'] == ticker) &
        (df['BioGuideID'] != politician)
    )
    window_trades = df[window_mask]
    
    # 4.1 Number of other politicians
    n_other_politicians = window_trades['BioGuideID'].nunique()
    
    # 4.2 Same direction
    same_direction = window_trades[window_trades['is_purchase'] == is_buy]
    n_same_direction = same_direction['BioGuideID'].nunique()
    
    # 4.3 Same committee
    same_committee = window_trades[window_trades['committee_id'] == committee]
    n_same_committee = same_committee['BioGuideID'].nunique()
    
    # 4.4 Same party
    same_party = window_trades[window_trades['Party'] == party]
    n_same_party = same_party['BioGuideID'].nunique()
    
    # 4.5 Is first mover
    prior_trades = df[
        (df['trade_date'] >= window_start) & 
        (df['trade_date'] < trade_date) &
        (df['Ticker'] == ticker)
    ]
    is_first_mover = 1 if len(prior_trades) == 0 and n_other_politicians > 0 else 0
    
    coordination_features.append({
        'idx': idx,
        'n_politicians_same_ticker_window': n_other_politicians,
        'n_politicians_same_direction': n_same_direction,
        'n_same_committee_same_ticker': n_same_committee,
        'n_same_party_same_ticker': n_same_party,
        'is_first_mover': is_first_mover
    })
    
    if idx % 10000 == 0:
        print(f"      Processing {idx:,}/{len(df):,}...")

coord_df = pd.DataFrame(coordination_features).set_index('idx')
df = df.join(coord_df)

# 4.6 Coordination flags
df['is_coordinated'] = (df['n_politicians_same_ticker_window'] >= 2).astype(int)
df['is_highly_coordinated'] = (df['n_politicians_same_ticker_window'] >= 5).astype(int)

# 4.7 Coordination ratio
df['coordination_ratio'] = (
    df['n_politicians_same_direction'] / 
    df['n_politicians_same_ticker_window'].replace(0, np.nan)
).fillna(0)

print(f"   ✓ n_politicians_same_ticker_window, n_politicians_same_direction")
print(f"   ✓ n_same_committee_same_ticker, n_same_party_same_ticker")
print(f"   ✓ is_first_mover, is_coordinated, is_highly_coordinated")

# ============================================================================
# 5. TIMING FEATURES
# ============================================================================
print("\n[5/8] Timing features...")

df['trade_dow'] = df['trade_date'].dt.dayofweek
df['is_friday'] = (df['trade_dow'] == 4).astype(int)
df['trade_month'] = df['trade_date'].dt.month
df['trade_quarter'] = df['trade_date'].dt.quarter
df['is_quarter_end'] = df['trade_date'].dt.is_quarter_end.astype(int)
df['is_december'] = (df['trade_month'] == 12).astype(int)
df['days_to_year_end'] = (
    pd.to_datetime(df['trade_date'].dt.year.astype(str) + '-12-31') - df['trade_date']
).dt.days

print(f"   ✓ trade_dow, is_friday, trade_month, trade_quarter")
print(f"   ✓ is_quarter_end, is_december, days_to_year_end")

# ============================================================================
# 6. MARKET CONTEXT FEATURES
# ============================================================================
print("\n[6/8] Market context features...")

vol_75 = df['realized_vol_30d'].quantile(0.75)
df['is_high_vol_period'] = (df['realized_vol_30d'] > vol_75).astype(int)

illiq_75 = df['amihud_illiq_20d'].quantile(0.75)
df['is_illiquid_stock'] = (df['amihud_illiq_20d'] > illiq_75).astype(int)

if 'market_cap' in df.columns:
    cap_25 = df['market_cap'].quantile(0.25)
    df['is_small_cap'] = (df['market_cap'] < cap_25).astype(int)
else:
    df['is_small_cap'] = np.nan

df['is_high_beta'] = (df['beta_252d'] > 1.5).astype(int)
df['has_abnormal_volume'] = (df['volume_ratio_30d'] > 2).astype(int)

print(f"   ✓ is_high_vol_period, is_illiquid_stock, is_small_cap")
print(f"   ✓ is_high_beta, has_abnormal_volume")

# ============================================================================
# 7. ADDITIONAL ADVANCED FEATURES
# ============================================================================
print("\n[7/8] Additional advanced features...")

# 7.1 Committee-Sector Match (placeholder)
df['committee_sector_match'] = 0

# 7.2 Congressional Session
df['is_during_recess'] = df['trade_month'].isin([8, 12, 1]).astype(int)
df['is_during_session'] = 1 - df['is_during_recess']

# 7.3 Bipartisan Coordination
print("      Calculating bipartisan coordination...")

bipartisan_features = []
for idx, row in df.iterrows():
    trade_date = row['trade_date']
    ticker = row['Ticker']
    politician = row['BioGuideID']
    
    window_start = trade_date - timedelta(days=COORDINATION_WINDOW_DAYS)
    window_end = trade_date + timedelta(days=COORDINATION_WINDOW_DAYS)
    
    window_mask = (
        (df['trade_date'] >= window_start) & 
        (df['trade_date'] <= window_end) &
        (df['Ticker'] == ticker) &
        (df['BioGuideID'] != politician)
    )
    window_trades = df[window_mask]
    
    parties_in_window = window_trades['Party'].unique()
    has_dem = 'D' in parties_in_window
    has_rep = 'R' in parties_in_window
    is_bipartisan = 1 if (has_dem and has_rep) else 0
    
    n_dem = window_trades[window_trades['Party'] == 'D']['BioGuideID'].nunique()
    n_rep = window_trades[window_trades['Party'] == 'R']['BioGuideID'].nunique()
    
    bipartisan_features.append({
        'idx': idx,
        'is_bipartisan_coordination': is_bipartisan,
        'n_democrats_same_ticker': n_dem,
        'n_republicans_same_ticker': n_rep
    })
    
    if idx % 10000 == 0:
        print(f"      Processing {idx:,}/{len(df):,}...")

bipartisan_df = pd.DataFrame(bipartisan_features).set_index('idx')
df = df.join(bipartisan_df)

# 7.4 Disclosure Delay vs Peers
print("      Calculating disclosure delay vs peers...")

df['filed_week'] = df['Filed'].dt.to_period('W')
weekly_avg_delay = df.groupby('filed_week')['disclosure_delay'].transform('mean')
weekly_std_delay = df.groupby('filed_week')['disclosure_delay'].transform('std').replace(0, 1)

df['disclosure_delay_vs_peers'] = (df['disclosure_delay'] - weekly_avg_delay) / weekly_std_delay
df['disclosure_delay_vs_peers'] = df['disclosure_delay_vs_peers'].fillna(0)
df['disclosure_delay_percentile'] = df.groupby('filed_week')['disclosure_delay'].rank(pct=True)

# 7.5 Pre-Legislation Trade Flag
df['potential_pre_legislation'] = (
    (df['is_high_info_committee'] == 1) &
    (df['is_during_session'] == 1) &
    (df['is_coordinated'] == 1)
).astype(int)

print(f"   ✓ committee_sector_match, is_during_session, is_during_recess")
print(f"   ✓ is_bipartisan_coordination, n_democrats/republicans_same_ticker")
print(f"   ✓ disclosure_delay_vs_peers, disclosure_delay_percentile")
print(f"   ✓ potential_pre_legislation")

# ============================================================================
# 8. COMPOSITE SUSPICION SCORES
# ============================================================================
print("\n[8/8] Composite suspicion scores...")

# 8.1 Information Advantage Score
df['info_advantage_score'] = (
    df['is_senator'] * 1 +
    df['is_chair_or_ranking'] * 2 +
    df['is_high_info_committee'] * 1 +
    (df['Years in position'] > 10).astype(int) * 1
)

# 8.2 Trade Suspicion Score
df['trade_suspicion_score'] = (
    df['is_large_trade'] * 1 +
    df['is_late_disclosure'] * 2 +
    df['is_contrarian'] * 1 +
    df['is_new_ticker'] * 1 +
    (df['trade_size_vs_history'] > 2).fillna(False).astype(int) * 1
)

# 8.3 Coordination Suspicion Score
df['coordination_suspicion_score'] = (
    df['is_coordinated'] * 1 +
    df['is_highly_coordinated'] * 2 +
    (df['n_same_committee_same_ticker'] >= 2).astype(int) * 2 +
    df['is_first_mover'] * 1
)

# 8.4 Overall Suspicion Score
df['overall_suspicion_score'] = (
    df['info_advantage_score'] + 
    df['trade_suspicion_score'] + 
    df['coordination_suspicion_score']
)

# 8.5 Enhanced Coordination Score
df['coordination_score_enhanced'] = (
    df['coordination_suspicion_score'] +
    df['is_bipartisan_coordination'] * 2 +
    (df['disclosure_delay_vs_peers'] > 1).astype(int) * 1
)

# 8.6 Overall Enhanced Score
df['overall_suspicion_score_enhanced'] = (
    df['info_advantage_score'] + 
    df['trade_suspicion_score'] + 
    df['coordination_score_enhanced'] +
    df['potential_pre_legislation'] * 2
)

print(f"   ✓ info_advantage_score, trade_suspicion_score")
print(f"   ✓ coordination_suspicion_score, overall_suspicion_score")
print(f"   ✓ coordination_score_enhanced, overall_suspicion_score_enhanced")

# ============================================================================
# SUMMARY AND EXPORT
# ============================================================================

new_features = [
    # Politician characteristics (5)
    'is_senator', 'committee_role_power', 'is_chair_or_ranking', 
    'is_high_info_committee', 'n_committees',
    
    # Trade characteristics (9)
    'disclosure_delay', 'is_late_disclosure', 'trade_size_mid',
    'is_large_trade', 'is_very_large_trade', 'is_purchase', 'is_full_sale',
    'is_contrarian', 'is_strong_contrarian',
    
    # Historical behavior (10)
    'politician_trade_count', 'days_since_last_trade',
    'politician_rolling_win_rate', 'politician_rolling_avg_car',
    'politician_6m_trade_count', 'politician_avg_monthly_trades',
    'politician_avg_trade_size', 'trade_size_vs_history',
    'ticker_trade_count', 'is_new_ticker',
    
    # Coordination (8)
    'n_politicians_same_ticker_window', 'n_politicians_same_direction',
    'n_same_committee_same_ticker', 'n_same_party_same_ticker',
    'is_first_mover', 'is_coordinated', 'is_highly_coordinated', 'coordination_ratio',
    
    # Timing (7)
    'trade_dow', 'is_friday', 'trade_month', 'trade_quarter',
    'is_quarter_end', 'is_december', 'days_to_year_end',
    
    # Market context (5)
    'is_high_vol_period', 'is_illiquid_stock', 'is_small_cap',
    'is_high_beta', 'has_abnormal_volume',
    
    # Advanced features (9)
    'committee_sector_match', 'is_during_session', 'is_during_recess',
    'is_bipartisan_coordination', 'n_democrats_same_ticker', 'n_republicans_same_ticker',
    'disclosure_delay_vs_peers', 'disclosure_delay_percentile',
    'potential_pre_legislation',
    
    # Composite scores (6)
    'info_advantage_score', 'trade_suspicion_score',
    'coordination_suspicion_score', 'overall_suspicion_score',
    'coordination_score_enhanced', 'overall_suspicion_score_enhanced'
]

print("\n" + "="*70)
print("FEATURE ENGINEERING COMPLETE")
print("="*70)
print(f"\nNew features created: {len(new_features)}")
print(f"Total columns: {len(df.columns)}")
print(f"Total trades: {len(df):,}")

print("\n" + "-"*70)
print("NEW FEATURES SUMMARY:")
print("-"*70)
for feat in new_features:
    if feat in df.columns:
        non_null = df[feat].notna().sum()
        pct = non_null / len(df) * 100
        print(f"  {feat:<40} {pct:>6.1f}% valid")

# Save
df.to_parquet(OUTPUT_PATH, index=False)
print(f"\n✅ Saved to: {OUTPUT_PATH}")

print("\n" + "="*70)
print("DONE!")
print("="*70)

FEATURE ENGINEERING FOR ANOMALY DETECTION

Loaded: 99,609 trades

[1/8] Politician characteristics...
   ✓ is_senator, is_chair_or_ranking, is_high_info_committee, n_committees

[2/8] Trade characteristics...
   ✓ disclosure_delay, is_late_disclosure, trade_size_mid
   ✓ is_large_trade, is_very_large_trade, is_purchase, is_full_sale
   ✓ is_contrarian, is_strong_contrarian

[3/8] Politician historical behavior...
   ✓ politician_trade_count, days_since_last_trade
   ✓ politician_rolling_win_rate, politician_rolling_avg_car
   ✓ politician_6m_trade_count, trade_size_vs_history, is_new_ticker

[4/8] Coordination features...
      This may take a few minutes...
      Processing 0/99,609...
      Processing 10,000/99,609...
      Processing 20,000/99,609...
      Processing 30,000/99,609...
      Processing 40,000/99,609...
      Processing 50,000/99,609...
      Processing 60,000/99,609...
      Processing 70,000/99,609...
      Processing 80,000/99,609...
      Processing 90,000/99,609..

In [ ]:
"""
================================================================================
DOWNLOAD SECTORS FROM YFINANCE AND MERGE TO ANOMALY DATASET
================================================================================

"""

import yfinance as yf
from tqdm import tqdm
import time

# ============================================================================
# CONFIGURATION
# ============================================================================

ANOMALY_FEATURES_PATH = r'C:\Users\sebib\Documents\GitHub\US_Congress\data\congress_trades\congress_trades_anomaly_features.parquet'
OUTPUT_PATH = r'C:\Users\sebib\Documents\GitHub\US_Congress\data\congress_trades\congress_trades_anomaly_features.parquet'  # Overwrite

# ============================================================================
# LOAD DATA
# ============================================================================

print("="*70)
print("DOWNLOADING SECTORS FROM YFINANCE")
print("="*70)

df = pd.read_parquet(ANOMALY_FEATURES_PATH)
print(f"\nLoaded: {len(df):,} trades")

# Get unique tickers
tickers = df['Ticker'].dropna().unique()
print(f"Unique tickers: {len(tickers):,}")

# ============================================================================
# DOWNLOAD SECTORS
# ============================================================================

print("\nDownloading sector info from yfinance...")
print("This may take 10-20 minutes...\n")

sector_data = []
errors = []

for i, ticker in enumerate(tqdm(tickers, desc="Downloading")):
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        if info and isinstance(info, dict):
            sector_data.append({
                'Ticker': ticker,
                'sector': info.get('sector', None),
                'industry': info.get('industry', None),
                'longName': info.get('longName', None)
            })
        else:
            sector_data.append({
                'Ticker': ticker,
                'sector': None,
                'industry': None,
                'longName': None
            })
            
    except Exception as e:
        errors.append({'Ticker': ticker, 'error': str(e)})
        sector_data.append({
            'Ticker': ticker,
            'sector': None,
            'industry': None,
            'longName': None
        })
    
    # Rate limiting - be nice to Yahoo
    if i % 100 == 0 and i > 0:
        time.sleep(1)

# ============================================================================
# CREATE SECTOR DATAFRAME
# ============================================================================

sector_df = pd.DataFrame(sector_data)

print(f"\n✅ Downloaded: {len(sector_df):,} tickers")
print(f"   With sector: {sector_df['sector'].notna().sum():,} ({sector_df['sector'].notna().mean():.1%})")
print(f"   Errors: {len(errors):,}")

# Show sector distribution
print("\nSector distribution:")
print(sector_df['sector'].value_counts().head(15))

# ============================================================================
# MERGE TO MAIN DATASET
# ============================================================================

print("\nMerging to anomaly dataset...")

# Merge
df = df.merge(sector_df[['Ticker', 'sector', 'industry']], on='Ticker', how='left')

print(f"   Trades with sector: {df['sector'].notna().sum():,} ({df['sector'].notna().mean():.1%})")

# ============================================================================
# CREATE SECTOR-BASED FEATURES
# ============================================================================

print("\nCreating sector-based features...")

# 1. Committee-Sector Match (ya existía pero ahora con datos reales)
committee_sector_map = {
    'finance': ['Financial Services', 'Financial', 'Banks'],
    'banking': ['Financial Services', 'Financial', 'Banks'],
    'energy': ['Energy', 'Utilities'],
    'health': ['Healthcare', 'Biotechnology'],
    'commerce': ['Consumer Cyclical', 'Consumer Defensive', 'Retail'],
    'technology': ['Technology', 'Communication Services'],
    'transportation': ['Industrials'],
    'agriculture': ['Consumer Defensive', 'Basic Materials'],
    'armed services': ['Industrials', 'Aerospace & Defense'],
    'intelligence': ['Technology', 'Industrials'],
}

def check_sector_match(row):
    """Check if politician's committee is related to stock's sector"""
    committee_topic = str(row.get('committee_topic', '')).lower()
    sector = row.get('sector', '')
    
    if pd.isna(sector) or pd.isna(committee_topic):
        return 0
    
    relevant_sectors = committee_sector_map.get(committee_topic, [])
    
    for rel_sector in relevant_sectors:
        if rel_sector.lower() in sector.lower():
            return 1
    return 0

df['committee_sector_match'] = df.apply(check_sector_match, axis=1)
print(f"   committee_sector_match: {df['committee_sector_match'].sum():,} ({df['committee_sector_match'].mean():.1%})")

# 2. Is High-Regulation Sector (más expuestos a decisiones políticas)
high_regulation_sectors = ['Healthcare', 'Financial Services', 'Energy', 'Utilities', 'Communication Services']
df['is_regulated_sector'] = df['sector'].isin(high_regulation_sectors).astype(int)
print(f"   is_regulated_sector: {df['is_regulated_sector'].sum():,} ({df['is_regulated_sector'].mean():.1%})")

# 3. Is Defense/Government Sector (contratos gubernamentales)
defense_sectors = ['Industrials', 'Aerospace & Defense']
defense_industries = ['Aerospace & Defense', 'Defense', 'Government']
df['is_defense_sector'] = (
    df['sector'].isin(defense_sectors) | 
    df['industry'].str.contains('Defense|Aerospace|Government', case=False, na=False)
).astype(int)
print(f"   is_defense_sector: {df['is_defense_sector'].sum():,} ({df['is_defense_sector'].mean():.1%})")

# 4. Is Tech Sector (alta volatilidad, mucha atención)
df['is_tech_sector'] = (df['sector'] == 'Technology').astype(int)
print(f"   is_tech_sector: {df['is_tech_sector'].sum():,} ({df['is_tech_sector'].mean():.1%})")

# 5. Is Healthcare/Biotech (relevante para comités de salud)
df['is_healthcare_sector'] = (
    (df['sector'] == 'Healthcare') | 
    df['industry'].str.contains('Biotech|Pharma|Drug', case=False, na=False)
).astype(int)
print(f"   is_healthcare_sector: {df['is_healthcare_sector'].sum():,} ({df['is_healthcare_sector'].mean():.1%})")

# 6. Is Finance Sector
df['is_finance_sector'] = (df['sector'] == 'Financial Services').astype(int)
print(f"   is_finance_sector: {df['is_finance_sector'].sum():,} ({df['is_finance_sector'].mean():.1%})")

# 7. Sector Concentration per Politician (HHI)
# ¿El político se concentra en pocos sectores o diversifica?
print("   Calculating sector concentration per politician...")

def calc_sector_hhi(group):
    """Calculate Herfindahl-Hirschman Index for sector concentration"""
    if len(group) < 5:
        return np.nan
    sector_counts = group['sector'].value_counts(normalize=True)
    hhi = (sector_counts ** 2).sum()
    return hhi

politician_sector_hhi = df.groupby('BioGuideID').apply(calc_sector_hhi).reset_index()
politician_sector_hhi.columns = ['BioGuideID', 'politician_sector_hhi']
df = df.merge(politician_sector_hhi, on='BioGuideID', how='left')
print(f"   politician_sector_hhi: {df['politician_sector_hhi'].notna().sum():,} valid")

# 8. Is New Sector for Politician (primera vez que tradea en este sector)
print("   Calculating new sector trades...")
df = df.sort_values(['BioGuideID', 'trade_date']).reset_index(drop=True)

def mark_new_sector(group):
    group = group.sort_values('trade_date')
    seen_sectors = set()
    is_new = []
    for sector in group['sector']:
        if pd.isna(sector):
            is_new.append(0)
        elif sector not in seen_sectors:
            is_new.append(1)
            seen_sectors.add(sector)
        else:
            is_new.append(0)
    group['is_new_sector'] = is_new
    return group

df = df.groupby('BioGuideID', group_keys=False).apply(mark_new_sector)
print(f"   is_new_sector: {df['is_new_sector'].sum():,} ({df['is_new_sector'].mean():.1%})")

# 9. Sector Anomaly Rate (¿este sector tiene históricamente más anomalías?)
# Lo calculamos después del análisis de anomalías, pero preparamos la columna
df['sector_clean'] = df['sector'].fillna('Unknown')

# 10. N Politicians Same Sector Same Window
# ¿Cuántos políticos tradean el mismo SECTOR (no ticker) en la misma ventana?
# Esto es más amplio que coordinación por ticker
print("   Calculating sector-level coordination...")

# Esto es costoso, lo hacemos de forma simplificada
sector_week_counts = df.groupby(['sector_clean', pd.Grouper(key='trade_date', freq='W')])['BioGuideID'].transform('nunique')
df['n_politicians_same_sector_week'] = sector_week_counts
print(f"   n_politicians_same_sector_week: mean={df['n_politicians_same_sector_week'].mean():.1f}")

print("\n✅ Sector features created!")

# ============================================================================
# SAVE
# ============================================================================

df.to_parquet(OUTPUT_PATH, index=False)
print(f"\n✅ Saved to: {OUTPUT_PATH}")

# Also save sector mapping for reference
sector_df.to_csv(OUTPUT_PATH.replace('.parquet', '_sector_mapping.csv'), index=False)
print(f"✅ Sector mapping saved")

print("\n" + "="*70)
print("DONE!")
print("="*70)

# ============================================================================
# SUMMARY
# ============================================================================

print("\nNew columns added:")
print("  - sector: Stock sector (e.g., 'Technology', 'Healthcare')")
print("  - industry: Stock industry (more specific)")
print("  - committee_sector_match: 1 if politician's committee relates to stock's sector")
print("  - is_regulated_sector: 1 if Healthcare, Finance, Energy, Utilities, Telecom")
print("  - is_defense_sector: 1 if Aerospace/Defense (government contracts)")
print("  - is_tech_sector: 1 if Technology")
print("  - is_healthcare_sector: 1 if Healthcare/Biotech/Pharma")
print("  - is_finance_sector: 1 if Financial Services")
print("  - politician_sector_hhi: Sector concentration (0-1, higher = more concentrated)")
print("  - is_new_sector: 1 if first time politician trades in this sector")
print("  - n_politicians_same_sector_week: # politicians trading same sector that week")

print(f"\nTotal features now: {len(df.columns)}")

DOWNLOADING SECTORS FROM YFINANCE

Loaded: 99,609 trades
Unique tickers: 4,683

This may take 10-20 minutes...



Downloading:   1%|          | 53/4683 [00:38<54:13,  1.42it/s]  